# SP-1 — train Conv-LSTM 1-step OHLC predictor

**Goal:** train a Conv-LSTM on Binance OHLCV historical data, evaluate on 5 fixed regime windows, save a checkpoint that the trading-radar backend can activate.

**Data path (pick ONE before running):**
- **Drive:** mount Google Drive, expect Parquet files at `/content/drive/MyDrive/trading-radar/ml-exports/`. Cell 2 below.
- **B2:** install boto3, set `B2_KEY_ID` + `B2_APP_KEY` + `B2_BUCKET` env vars, pulls from `b2://trading-radar-backups/ml-exports/`. Cell 3 below.

**Output:** a checkpoint `.pt` file uploaded back to Drive/B2 + the eval JSON. Then call `POST /api/v1/admin/ml-checkpoints` from your laptop to register it.

Spec reference: `docs/superpowers/specs/2026-05-05-SP-1-ml-data-ghost-candles-design.md` §3, §5, §6.1.

## 1. Install dependencies

Colab T4 free tier already has torch + pandas + numpy. Add pyarrow for Parquet.

In [ ]:
!pip install -q pyarrow duckdb boto3

## 2. (Drive option) Mount Google Drive

Skip this cell if using B2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/trading-radar/ml-exports'
!ls -lh "$DATA_ROOT"

## 3. (B2 option) Pull Parquet from Backblaze

Skip this cell if using Drive. Set B2 env vars below before running.

In [ ]:
import os, boto3
os.environ['B2_KEY_ID'] = ''   # paste key id
os.environ['B2_APP_KEY'] = ''  # paste app key
B2_BUCKET = 'trading-radar-backups'
B2_ENDPOINT = 'https://s3.us-west-002.backblazeb2.com'
DATA_ROOT = '/content/data'
!mkdir -p $DATA_ROOT
s3 = boto3.client('s3', endpoint_url=B2_ENDPOINT,
                  aws_access_key_id=os.environ['B2_KEY_ID'],
                  aws_secret_access_key=os.environ['B2_APP_KEY'])
for key in ['ohlcv_full.parquet']:
    s3.download_file(B2_BUCKET, f'ml-exports/{key}', f'{DATA_ROOT}/{key}')
!ls -lh $DATA_ROOT

## 4. Load OHLCV + build training pairs

Sliding window: 256 input bars → predict next 1 bar. Time-based train/val/test split.

In [ ]:
import duckdb, pandas as pd, numpy as np, torch
from torch.utils.data import DataLoader, TensorDataset

WINDOW = 256
TRAIN_END = '2023-12-31'
VAL_END   = '2024-12-31'

df = duckdb.sql(f"""
  SELECT symbol, ts, open, high, low, close, volume
  FROM parquet_scan('{DATA_ROOT}/ohlcv_full.parquet')
  WHERE timeframe='1h'
  ORDER BY symbol, ts
""").df()
print(f'rows: {len(df):,}, symbols: {df.symbol.nunique()}, span: {df.ts.min()} .. {df.ts.max()}')

def build_windows(group: pd.DataFrame):
    arr = group[['open','high','low','close','volume']].to_numpy(dtype=np.float32)
    if len(arr) < WINDOW + 1:
        return [], []
    X, Y = [], []
    for i in range(WINDOW, len(arr)):
        window = arr[i-WINDOW:i]
        last_close = window[-1, 3]
        if last_close <= 0:
            continue
        x_pct = window[:, :4] / last_close - 1.0  # OHLC as % change
        vol_z = (window[:, 4] - window[:, 4].mean()) / (window[:, 4].std() + 1e-9)
        x = np.column_stack([x_pct, vol_z]).astype(np.float32)
        y = (arr[i, :4] / last_close - 1.0).astype(np.float32)
        X.append(x); Y.append(y)
    return X, Y

all_X, all_Y, all_ts = [], [], []
for sym, g in df.groupby('symbol'):
    g = g.sort_values('ts').reset_index(drop=True)
    Xs, Ys = build_windows(g)
    if Xs:
        all_X += Xs; all_Y += Ys
        all_ts += list(g.ts.iloc[WINDOW:WINDOW+len(Xs)])

X = np.stack(all_X); Y = np.stack(all_Y); TS = pd.to_datetime(all_ts)
train_mask = TS <= TRAIN_END
val_mask   = (TS > TRAIN_END) & (TS <= VAL_END)
test_mask  = TS > VAL_END
print(f'train: {train_mask.sum():,}  val: {val_mask.sum():,}  test: {test_mask.sum():,}')

## 5. Define model (matches `app/ml/model.py`)

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class ConvLSTMPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(5, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv_drop = nn.Dropout(0.2)
        self.lstm = nn.LSTM(64, 128, num_layers=2, dropout=0.2, batch_first=True)
        self.fc = nn.Linear(128, 4)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv_drop(x)
        x = x.transpose(1, 2)
        h, _ = self.lstm(x)
        return self.fc(h[:, -1, :])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ConvLSTMPredictor().to(device)
print(f'device: {device}, params: {sum(p.numel() for p in model.parameters()):,}')

## 6. Train

MAE loss + Adam + cosine LR decay + early stopping.

In [ ]:
BATCH = 64
EPOCHS = 30
PATIENCE = 5

X_train_t = torch.from_numpy(X[train_mask])
Y_train_t = torch.from_numpy(Y[train_mask])
X_val_t   = torch.from_numpy(X[val_mask])
Y_val_t   = torch.from_numpy(Y[val_mask])

train_dl = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_val_t, Y_val_t),   batch_size=BATCH, shuffle=False)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
best_val = float('inf'); best_state = None; epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = (pred - yb).abs().mean()
        opt.zero_grad(); loss.backward(); opt.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(X_train_t)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += (model(xb) - yb).abs().mean().item() * len(xb)
    val_loss /= len(X_val_t)

    sched.step()
    print(f'epoch {epoch+1:2d}: train MAE {train_loss*100:.3f}%  val MAE {val_loss*100:.3f}%')
    if val_loss < best_val:
        best_val = val_loss; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('early stop'); break

print(f'best val MAE: {best_val*100:.3f}%')
model.load_state_dict(best_state)

## 7. Evaluate on the 5 regime windows

All 5 must show MAE ≤ 1.5% to pass acceptance (spec §13).

In [ ]:
REGIMES = [
    ('bull_breakout',   '2020-10-01', '2021-04-30'),
    ('bear_crash',      '2022-04-01', '2022-12-31'),
    ('sideways_grind',  '2023-04-01', '2023-09-30'),
    ('high_volatility', '2020-03-01', '2020-04-15'),
    ('low_volatility',  '2024-04-01', '2024-07-31'),
]
ACCEPTANCE = 0.015

model.eval()
results = {}
btc = df[df.symbol == 'BTC/USDT'].sort_values('ts').reset_index(drop=True)
for name, start, end in REGIMES:
    sub = btc[(btc.ts >= start) & (btc.ts <= end)].reset_index(drop=True)
    if len(sub) < WINDOW + 1:
        results[name] = {'mae': None, 'samples': 0, 'passes': False}
        continue
    Xs, Ys = build_windows(sub)
    if not Xs:
        results[name] = {'mae': None, 'samples': 0, 'passes': False}
        continue
    Xt = torch.from_numpy(np.stack(Xs)).to(device)
    Yt = torch.from_numpy(np.stack(Ys)).to(device)
    with torch.no_grad():
        preds = model(Xt)
        mae = (preds - Yt).abs().mean().item()
    results[name] = {'mae': mae, 'samples': len(Xs), 'passes': mae <= ACCEPTANCE}
    print(f'{name:18s}: MAE {mae*100:.3f}%  samples {len(Xs):,}  passes {results[name]["passes"]}')

all_pass = all(r['passes'] for r in results.values())
print(f'\nALL PASS: {all_pass}')

## 8. Save checkpoint + register via admin API

Only proceed if `all_pass == True`. Otherwise re-tune and rerun cells 5-7.

In [ ]:
import hashlib, json, time

VERSION = '0.1.0'
OUT = f'/content/conv_lstm_v{VERSION}.pt'
torch.save(model.state_dict(), OUT)
sha256 = hashlib.sha256(open(OUT, 'rb').read()).hexdigest()
print(f'saved: {OUT}  sha256: {sha256}')

# Upload back to Drive (or B2) — example for Drive:
import shutil
shutil.copy(OUT, f'{DATA_ROOT}/conv_lstm_v{VERSION}.pt')
print(f'uploaded to {DATA_ROOT}/conv_lstm_v{VERSION}.pt')

registration = {
    'model_name': 'conv_lstm_predictor',
    'version': VERSION,
    'checkpoint_uri': f'file:///app/data/ml-checkpoints/conv_lstm_v{VERSION}.pt',  # adjust for your prod path
    'sha256': sha256,
    'trained_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'train_data_window': '2017-01 to 2023-12 (val 2024)',
    'eval_results': {k: v['mae'] for k, v in results.items() if v['mae'] is not None},
    'notes': 'Auto-trained from Colab notebook',
}
print(json.dumps(registration, indent=2))
print('\nNext step (run on your laptop):')
print(f'curl -X POST https://trading-radar.your-domain/api/v1/admin/ml-checkpoints -H "Content-Type: application/json" -d @registration.json')
print('Then activate via:')
print('curl -X PATCH https://.../admin/ml-checkpoints/{id} -d \'{"is_active": true}\'')